# Bag of Chips — SRM reconstruction on Colab
This notebook runs the corrected SRM implementation on the authors' real **Cans** dataset. Select **Runtime → Change runtime type → T4 GPU** before starting. Run cells from top to bottom.

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU found. Select Runtime > Change runtime type > T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)

## 1. Upload the source bundle
Upload `cs180_colab_source.zip`, supplied alongside this notebook. It contains code only; the real dataset is downloaded separately below.

In [ ]:
from google.colab import files
from pathlib import Path
import shutil
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith('.zip')), None)
assert archive, 'Please upload cs180_colab_source.zip'
project = Path('/content/cs180_final_project')
project.mkdir(exist_ok=True)
shutil.unpack_archive(archive, project)
%cd /content/cs180_final_project
print('Project files:', sorted(p.name for p in project.glob('*.py')))

In [ ]:
!pip -q install numpy opencv-python-headless matplotlib embreex
import torchvision
print('PyTorch', torch.__version__, '| torchvision', torchvision.__version__)

## 2. Download and preprocess the real Cans dataset
This downloads the official data from the authors. Preprocessing creates the reflection direction, visibility, diffuse, depth, normal, and first-bounce buffers. It only needs to run once per Colab session.

In [ ]:
from pathlib import Path
if not Path('cans_data/xy_mesh.ply').exists():
    !wget -q --show-progress https://grail.cs.washington.edu/projects/bagofchips/data/cans.zip -O /content/cans.zip
    !unzip -q /content/cans.zip -d /content/cans_download
    import os
    candidates = list(Path('/content/cans_download').rglob('xy_mesh.ply'))
    assert candidates, 'The downloaded archive did not contain xy_mesh.ply'
    source = candidates[0].parent
    if Path('cans_data').exists(): shutil.rmtree('cans_data')
    shutil.move(str(source), 'cans_data')
print('Dataset ready:', len(list(Path('cans_data/rgb').glob('*.png'))), 'RGB frames')

In [ ]:
if not Path('cache/meta.json').exists():
    !python prepare.py
else:
    print('Using existing prepared cache')

## 3. Store checkpoints in Google Drive
This prevents losing training progress when the Colab runtime disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_runs = Path('/content/drive/MyDrive/bagofchips_runs')
drive_runs.mkdir(parents=True, exist_ok=True)
runs = Path('/content/cs180_final_project/runs')
if runs.is_symlink():
    runs.unlink()
elif runs.exists():
    backup = Path('/content/cs180_final_project/runs_local_backup')
    if not backup.exists(): runs.rename(backup)
runs.symlink_to(drive_runs, target_is_directory=True)
print('Checkpoints:', runs.resolve())

## 4. Train the SRM
`QUICK_TEST=True` runs only 50 updates to verify the pipeline. It will **not** produce a meaningful panorama. Set it to `False` for the paper's 40-epoch configuration. A T4 may still require several hours.

In [ ]:
QUICK_TEST = True
RUN_NAME = 'cans_colab_test' if QUICK_TEST else 'cans_paper_corrected'
extra = '--max-steps 50 --preview 25 --log 10' if QUICK_TEST else '--epochs 40 --preview 1000 --log 100'
cmd = (f'python train.py srm --run {RUN_NAME} --materials 2 --cans all '
       f'--batch 2 --lr-srm 0.001 --lambda-p 1 --lambda-g 0.05 {extra}')
print(cmd)
!{cmd}

To continue an interrupted full run, keep the same `RUN_NAME` and execute the command below. Do not use `--resume` after changing architecture, material count, or loss settings.

In [ ]:
# Uncomment only when resuming a compatible interrupted run.
# !python train.py srm --run cans_paper_corrected --materials 2 --cans all --epochs 40 --batch 2 --lr-srm 0.001 --lambda-p 1 --lambda-g 0.05 --resume --log 100

## 5. Evaluate and display the reconstructed panorama
The paper-style image is a 500×175 crop without the blue coverage overlay.

In [ ]:
!python eval_env.py --run {RUN_NAME} --paper-figure
from IPython.display import Image, display
result = Path('results') / RUN_NAME / 'pano_paper.png'
display(Image(filename=str(result)))

In [ ]:
output_zip = shutil.make_archive(f'/content/{RUN_NAME}_results', 'zip', Path('results') / RUN_NAME)
files.download(output_zip)